# Floating-point arithmetic<a href="#Floating-point-arithmetic" class="anchor-link">¶</a>

As a data analyst, you will be concerned primarily with *numerical
programs* in which the bulk of the computational work involves
floating-point computation. This notebook guides you through some of the
most fundamental concepts in how computers store real numbers, so you
can be smarter about your number crunching.

## WYSInnWYG, or "what you see is not necessarily what you get."<a
href="#WYSInnWYG,-or-%22what-you-see-is-not-necessarily-what-you-get.%22"
class="anchor-link">¶</a>

One important consequence of a binary format is that when you print
values in base ten, what you see may not be what you get! For instance,
try running the code below.

> This code invokes Python's
> [`decimal`](https://docs.python.org/3/library/decimal.html) package,
> which implements base-10 floating-point arithmetic in software.

In \[1\]:

    from decimal import Decimal
    ?Decimal # Asks for a help page on the Decimal() constructor

In \[2\]:

    x = 1.0 + 2.0**(-52)

    print(x)
    print(Decimal(x)) # What does this do?

    print(Decimal(0.1) - Decimal('0.1')) # Why does the output appear as it does?

    1.0000000000000002
    1.0000000000000002220446049250313080847263336181640625
    5.551115123125782702118158340E-18

> Aside: If you ever need true decimal storage with no loss of precision
> (e.g., an accounting application), turn to the `decimal` package. Just
> be warned it might be slower. See the following experiment for a
> practical demonstration.

In \[3\]:

    from random import random

    NUM_TRIALS = 2500000

    print("Native arithmetic:")
    A_native = [random() for _ in range(NUM_TRIALS)]
    B_native = [random() for _ in range(NUM_TRIALS)]
    %timeit [a+b for a, b in zip(A_native, B_native)]

    print("\nDecimal package:")
    A_decimal = [Decimal(a) for a in A_native]
    B_decimal = [Decimal(b) for b in B_native]
    %timeit [a+b for a, b in zip(A_decimal, B_decimal)]

    Native arithmetic:
    212 ms ± 4.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

    Decimal package:
    583 ms ± 11.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

**The same and not the same.** Consider the following two program
fragments:

*Program 1*:

    s = a - b
    t = s + b

*Program 2*:

    s = a + b
    t = s - b

Let \$a=1.0\$ and \$b=\epsilon_d / 2 \approx 1.11 \times 10^{-16}\$,
i.e., machine epsilon for IEEE-754 double-precision. Recall that we do
not expect these programs to return the same value; let's run some
Python code to see.

> Note: The IEEE standard guarantees that given two finite-precision
> floating-point values, the result of applying any binary operator to
> them is the same as if the operator were applied in infinite-precision
> and then rounded back to finite-precision. The precise nature of
> rounding can be controlled by so-called *rounding modes*; the default
> rounding mode is
> "[round-half-to-even](http://en.wikipedia.org/wiki/Rounding)."

In \[4\]:

    a = 1.0
    b = 2.**(-53) # == $\epsilon_d$ / 2.0

    s1 =  a - b
    t1 = s1 + b

    s2 =  a + b
    t2 = s2 - b

    print("s1:", s1.hex())
    print("t1:", t1.hex())
    print("\n")
    print("s2:", s2.hex())
    print("t2:", t2.hex())

    print("")
    print(t1, "vs.", t2)
    print("(t1 == t2) == {}".format(t1 == t2))

    s1: 0x1.fffffffffffffp-1
    t1: 0x1.0000000000000p+0


    s2: 0x1.0000000000000p+0
    t2: 0x1.fffffffffffffp-1

    1.0 vs. 0.9999999999999999
    (t1 == t2) == False

By the way, the NumPy/SciPy package, which we will cover later in the
semester, allows you to determine machine epsilon in a portable way.
Just note this fact for now.

Here is an example of printing machine epsilon for both single-precision
and double-precision values.

In \[5\]:

    import numpy as np

    EPS_S = np.finfo (np.float32).eps
    EPS_D = np.finfo (float).eps

    print("Single-precision machine epsilon:", float(EPS_S).hex(), "~", EPS_S)
    print("Double-precision machine epsilon:", float(EPS_D).hex(), "~", EPS_D)

    Single-precision machine epsilon: 0x1.0000000000000p-23 ~ 1.1920929e-07
    Double-precision machine epsilon: 0x1.0000000000000p-52 ~ 2.220446049250313e-16

## Analyzing floating-point programs<a href="#Analyzing-floating-point-programs" class="anchor-link">¶</a>

Let's say someone devises an algorithm to compute \$f(x)\$. For a given
value \$x\$, let's suppose this algorithm produces the value
\$\mathrm{alg}(x)\$. One important question might be, is that output
"good" or "bad?"

**Forward stability.** One way to show that the algorithm is good is to
show that

\$\$ \left\| \mathrm{alg}(x) - f(x) \right\| \$\$

is "small" for all \$x\$ of interest to your application. What is small
depends on context. In any case, if you can show it then you can claim
that the algorithm is *forward stable*.

**Backward stability.** Sometimes it is not easy to show forward
stability directly. In such cases, you can also try a different
technique, which is to show that the algorithm is, instead, *backward
stable*.

In particular, \$\mathrm{alg}(x)\$ is a *backward stable algorithm* to
compute \$f(x)\$ if, for all \$x\$, there exists a "small" \$\Delta x\$
such that

\$\$\mathrm{alg}(x) = f(x + \Delta x).\$\$

In other words, if you can show that the algorithm produces the exact
answer to a slightly different input problem (meaning \$\Delta x\$ is
small, again in a context-dependent sense), then you can claim that the
algorithm is backward stable.

**Round-off errors.** You already know that numerical values can only be
represented finitely, which introduces round-off error. Thus, at the
very least we should hope that a scheme to compute \$f(x)\$ is as
insensitive to round-off errors as possible. In other words, given that
there will be round-off errors, if you can prove that
\$\mathrm{alg}(x)\$ is either forward or backward stable, then that will
give you some measure of confidence that your algorithm is good.

Here is the "standard model" of round-off error. Start by assuming that
every scalar floating-point operation incurs some bounded error. That
is, let \$a \odot b\$ be the exact mathematical result of some operation
on the inputs, \$a\$ and \$b\$, and let \$\mathrm{fl}(a \odot b)\$ be
the *computed* value, after rounding in finite-precision. The standard
model says that

\$\$\mathrm{fl}(a \odot b) \equiv (a \odot b) (1 + \delta),\$\$

where \$\|\delta\| \leq \epsilon\$, machine epsilon.

Let's apply these concepts on an example.

## Example: Computing a sum<a href="#Example:-Computing-a-sum" class="anchor-link">¶</a>

Let \$x \equiv (x_0, \ldots, x\_{n-1})\$ be a collection of input data
values. Suppose we wish to compute their sum.

The exact mathematical result is

\$\$ f(x) \equiv \sum\_{i=0}^{n-1} x_i. \$\$

Given \$x\$, let's also denote its exact sum by the synonym \$s\_{n-1}
\equiv f(x)\$.

Now consider the following Python program to compute its sum:

In \[6\]:

    def alg_sum(x): # x == x[:n]
        s = 0.
        for x_i in x: # x_0, x_1, \ldots, x_{n-1}
            s += x_i
        return s

In exact arithmetic, meaning without any rounding errors, this program
would compute the exact sum. (See also the note below.) However, you
know that finite arithmetic means there will be some rounding error
after each addition.

Let \$\delta_i\$ denote the (unknown) error at iteration \$i\$. Then,
assuming the collection `x` represents the input values exactly, you can
show that `alg_sum(x)` computes \$\hat{s}\_{n-1}\$ where

\$\$ \hat{s}\_{n-1} \approx s\_{n-1} + \sum\_{i=0}^{n-1} s_i \delta_i,
\$\$

that is, the exact sum *plus* a perturbation, which is the second term
(the sum). The question, then, is under what conditions will this sum
will be small?

Using a *backward error analysis*, you can show that

\$\$ \hat{s}\_{n-1} \approx \sum\_{i=0}^{n-1} x_i(1 + \Delta_i) = f(x +
\Delta), \$\$

where \$\Delta \equiv (\Delta_0, \Delta_1, \ldots, \Delta\_{n-1})\$. In
other words, the computed sum is the exact solution to a slightly
different problem, \$x + \Delta\$.

To complete the analysis, you can at last show that

\$\$ \|\Delta_i\| \leq (n-i) \epsilon, \$\$

where \$\epsilon\$ is machine precision. Thus, as long as \$n \epsilon
\ll 1\$, then the algorithm is backward stable and you should expect the
computed result to be close to the true result. Interpreted differently,
as long as you are summing \$n \ll \frac{1}{\epsilon}\$ values, then you
needn't worry about the accuracy of the computed result compared to the
true result:

In \[7\]:

    print("Single-precision: 1/epsilon_s ~= {:.1f} million".format(1e-6 / EPS_S))
    print("Double-precision: 1/epsilon_d ~= {:.1f} quadrillion".format(1e-15 / EPS_D))

    Single-precision: 1/epsilon_s ~= 8.4 million
    Double-precision: 1/epsilon_d ~= 4.5 quadrillion

Based on this result, you can probably surmise why double-precision is
usually the default in many languages.

In the case of this summation, we can quantify not just the *backward
error* (i.e., \$\Delta_i\$) but also the *forward error*. In that case,
it turns out that

\$\$ \left\| \hat{s}\_{n-1} - s\_{n-1} \right\| \lessapprox n \epsilon
\\\|x\\\|\_1. \$\$

> **Note: Analysis in exact arithmetic.** We claimed above that
> `alg_sum()` is correct *in exact arithmetic*, i.e., in the absence of
> round-off error. You probably have a good sense of that just reading
> the code.
>
> However, if you wanted to argue about its correctness more formally,
> you might do so as follows using the technique of [proof by
> induction](https://en.wikipedia.org/wiki/Mathematical_induction). When
> your loops are more complicated and you want to prove that they are
> correct, you can often adapt this technique to your problem.
>
> First, assume that the `for` loop enumerates each element `p[i]` in
> order from `i=0` to `n-1`, where `n=len(p)`. That is, assume `p_i` is
> `p[i]`.
>
> Let \$p_k \equiv \mathtt{p\[}k\mathtt{\]}\$ be the \$k\$-th element of
> `p[:]`. Let \$s_i \equiv \sum\_{k=0}^{i} p_k\$; in other words,
> \$s_i\$ is the *exact* mathematical sum of `p[:i+1]`. Thus,
> \$s\_{n-1}\$ is the exact sum of `p[:]`.
>
> Let \$\hat{s}\_{-1}\$ denote the initial value of the variable `s`,
> which is 0. For any \$i \geq 0\$, let \$\hat{s}\_i\$ denote the
> *computed* value of the variable `s` immediately after the execution
> of line 4, where \$i=\mathtt{i}\$. When \$i=\mathtt{i}=0\$,
> \$\hat{s}\_0 = \hat{s}\_{-1} + p_0 = p_0\$, which is the exact sum of
> `p[:1]`. Thus, \$\hat{s}\_0 = s_0\$.
>
> Now suppose that \$\hat{s}\_{i-1} = s\_{i-1}\$. When \$\mathtt{i}=i\$,
> we want to show that \$\hat{s}\_i = s_i\$. After line 4 executes,
> \$\hat{s}\_{i} = \hat{s}\_{i-1} + p_i = s\_{i-1} + p_i = s_i\$. Thus,
> the computed value \$\hat{s}\_i\$ is the exact sum \$s_i\$.
>
> If \$i=n\$, then, at line 5, the value
> \$\mathtt{s}=\hat{s}\_{n-1}=s\_{n-1}\$, and thus the program must in
> line 5 return the exact sum.

## A numerical experiment: Summation<a href="#A-numerical-experiment:-Summation" class="anchor-link">¶</a>

Let's do an experiment to verify that these bounds hold.

**Exercise 0** (2 points). In the code cell below, we've defined a list,

    N = [10, 100, 1000, 10000, 100000, 1000000, 10000000]

-   Take each entry `N[i]` to be a problem size.
-   Let `t[:len(N)]` be a list, which will hold computed sums.
-   For each `N[i]`, run an experiment where you sum a list of values
    `x[:N[i]]` using `alg_sum()`. You should initialize `x[:]` so that
    all elements have the value **`0.1`**. Store the computed sum in
    `t[i]`.

In \[8\]:

    N = [10, 100, 1000, 10000, 100000, 1000000, 10000000]

    # Initialize an array t of size len(N) to all zeroes.
    t = [0.0] * len(N)  

    # Your code should do the experiment described above for
    # each problem size N[i], and store the computed sum in t[i].

    ### BEGIN SOLUTION
    x = [0.1] * max(N)
    t = [alg_sum(x[0:n]) for n in N]
    ### END SOLUTION

    print(t)

    [0.9999999999999999, 9.99999999999998, 99.9999999999986, 1000.0000000001588, 10000.000000018848, 100000.00000133288, 999999.9998389754]

In \[9\]:

    # Test: `experiment_results`
    import pandas as pd
    from IPython.display import display

    import matplotlib.pyplot as plt
    %matplotlib inline

    s = [1., 10., 100., 1000., 10000., 100000., 1000000.] # exact sums
    t_minus_s_rel = [(t_i - s_i) / s_i for s_i, t_i in zip (s, t)]
    rel_err_computed = [abs(r) for r in t_minus_s_rel]
    rel_err_bound = [ni*EPS_D for ni in N]

    # Plot of the relative error bound
    plt.loglog (N, rel_err_computed, 'b*', N, rel_err_bound, 'r--')

    print("Relative errors in the computed result:")
    display (pd.DataFrame ({'n': N, 'rel_err': rel_err_computed, 'rel_err_bound': [n*EPS_D for n in N]}))

    assert all([abs(r) <= n*EPS_D for r, n in zip(t_minus_s_rel, N)])

    print("\n(Passed!)")

    Relative errors in the computed result:

|     | n        | rel_err      | rel_err_bound |
|-----|----------|--------------|---------------|
| 0   | 10       | 1.110223e-16 | 2.220446e-15  |
| 1   | 100      | 1.953993e-15 | 2.220446e-14  |
| 2   | 1000     | 1.406875e-14 | 2.220446e-13  |
| 3   | 10000    | 1.588205e-13 | 2.220446e-12  |
| 4   | 100000   | 1.884837e-12 | 2.220446e-11  |
| 5   | 1000000  | 1.332883e-11 | 2.220446e-10  |
| 6   | 10000000 | 1.610246e-10 | 2.220446e-09  |

    (Passed!)

![](data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAYQAAAEACAYAAACznAEdAAAABHNCSVQICAgIfAhkiAAAAAlwSFlz%0AAAALEgAACxIB0t1+/AAAADl0RVh0U29mdHdhcmUAbWF0cGxvdGxpYiB2ZXJzaW9uIDIuMS4yLCBo%0AdHRwOi8vbWF0cGxvdGxpYi5vcmcvNQv5yAAAIABJREFUeJzt3Xl81PWdx/HXRzRca6jFWlBQQJAV%0ArQpGtN1asfXASqS1unJYa6UiKq3V1QpqPUDFY63rgSAqxVoLsmzWxZOuVosoVKJ4QCkQgy7ByFVF%0AbkE++8d3KHFMYJI5fvObeT8fjzxmft/M8flCMp98b3N3RERE9og6ABERyQ9KCCIiAighiIhIghKC%0AiIgASggiIpKghCAiIoASgoiIJCghiIgIoIQgIiIJSggiIgLAnlEHkAozKwfK99577wsPOeSQqMMR%0AEYmVN954Y7W7f213j7M47WVUVlbmlZWVUYchIhIrZvaGu5ft7nHqMhIRESAmCcHMys1swtq1a6MO%0ARUSkYMUiIbj7U+4+tE2bNlGHIiJSsGKREEREJPtikRDUZSQikn2xSAjqMhIRyb5YJAQRkaKWo94R%0AJQQRkXy1dCn8+MfQrVtOkkIsViqLiBSdDRugZ0/YsgUuuwzMsv6WsWghaFBZRIrCp5/Cww+DO7Ru%0ADb/9LVRVwW23QWlp1t8+FglBg8oiUtC2bIH/+A84+GC48EKYNy+U//CHcMABOQsjFglBRKQgff45%0APPooHHIIXH45HHkkvP469OoVSTgaQxARicrmzTBiRGgFPPIInHRSpOFE2kIwsx5mNtXMxpnZWVHG%0AIiKSE7NmweDBsHVrGCd47bXQKog4GUAaCcHMJprZSjObn1Te18wWmVmVmY3YzcucBtzn7hcD5zU1%0AFhGRvPfuu1BeDscfDy+9BEuWhPLOnWGP/Oi9TyeKSUDfugVm1gwYS/ig7wEMTLQCvmFmTyd97Qc8%0ABgwwszuBtmnEIiKSn9atg/POC+MDr7wCY8aEmUM9ekQd2Zc0eQzB3WeaWaek4t5AlbtXA5jZFKC/%0Au48B+jXwUpcmEklFU2MREck7W7fCXnuFbqElS+DKK8N4wVe/GnVkDcr0oPIBwLI61zXAsQ09OJFQ%0ArgFaA3c28JihwFCAAw88MENhiohkybp1cNddMHEivP027LMPvPpq3nQL7Uqks4zc/X0SH/a7eMwE%0AM6sFyktKSo7OSWAiIo21ZQs8+CDcfDOsWgU/+hFs3BgSQgySAWQ+ISwHOta57pAoExEpXB9/HNYO%0AvP8+nHhiWFncu3fUUTVaptPWXKCbmXU2sxJgADA93RfVSmURyTvuMD8xyXKffeDss2HGDHjxxVgm%0AA0hv2ulkYDbQ3cxqzGyIu28DhgMzgIXAVHdfkG6Q2stIRPLKq6/Cd74TNp97771QdscdcMopOdmE%0ALlvM3aOOIWVlZWVeWVkZdRgiUqzmz4drroGnnoJ27eCGG2DIkDCbKI+Z2RvuXra7x8Vi6wozKwfK%0Au3btGnUoIlKs1qyBY46B5s3h1lvhF78IU0oLSCyGvjWGICKRWL06zBwCaNsWpkwJXUQjRxZcMoCY%0AJASNIYhITq1fD6NGQZcucMklsHhxKO/fPySGAhWLhKAWgojkxGefwf33h3MJbrgBTj4ZFiwI21MX%0AgViMIYiI5MSmTSERHHEETJ8Oxza40UJBikULQV1GIpIV7vDsszBwYDispk0beOst+NOfii4ZQEwS%0AgrqMRCTjZs+GE06A008P5xEsS2zD1rFjrNcSpCMWCUFEJGPWrIEf/AC+9a0wWDx2LCxcCJ06RR1Z%0A5GIxhqB1CCKSti1bwhqCNm2gtjZsQnfZZfBP/xR1ZHkjFi0EdRmJSJOtXg1XXBFmCq1fD3vuCXPm%0AwLXXKhkkiUVCEBFptPXrQyvg4IPhnnvge98Lh9pD0Y4R7E4suoxERBqltjZsPLdiRRgvuOWWvDyy%0AMt+ohSAihWH79jBlFKB9+3CO8ezZ8N//HftkUFsbJkR99FF23ycWCUHrEESkQe7w/PNw9NFw3HFQ%0AUxPK77gjXBeA0aNh1qywm0Y2xSIhaFBZROo1Z044oey00+DTT8M5xvvvH3VUGdOyZRjuGDcuNIDG%0AjQvXLVtm5/1ikRBERL6kpga+/e2whuD++8PtoEGxOb84FdXVoUqtWoXrVq1g8GBYujQ771c4/3Ii%0AUviWLQt/JgN06BDGB957Dy69FEpKoo0tC9q3h9LSMDmqRYtwW1oazubJhpwlBDPrYmaPmNm0OmWt%0AzexRM3vIzAbnKhYRiZk1a+DKK6FbN7j8cli+PJSXlxf8WoIVK2DYsNA7NmxYdgeWUzpC08wmAv2A%0Ale5+eJ3yvsA9QDPgYXe/LYXXmubuZyXu/xj4xN2fMrMn3P2cXT1XR2iKFJmNG+Huu8MA8fr1YebQ%0AjTfCQQdFHVmsZPoIzUnA/cDv6rxBM2AscDJQA8w1s+mE5DAm6fkXuPvKel63A/Bu4v7nKcYiIsVi%0Awwa4/Xb47nfDWoLDDos6ooKWUkJw95lm1impuDdQ5e7VAGY2Bejv7mMIrYlU1BCSwltoPENEtm+H%0AqVPhySdh8mT42tdg0aLQmS5Zl86H8AHAsjrXNYmyeplZWzMbD/Q0s5GJ4grgR2Y2DniqgecNNbNK%0AM6tctWpVGuGKSN5yhxkzoKwsnE2wcCGsTHQqKBnkTM62rnD3NcCwpLINwE9387wJZlYLlJeUlByd%0AxRBFJArLl8O558LLL4ctqB97rOCmj8ZFOv/iy4GOda47JMoyTgvTRArQxo3hdt99w/177w3dQ+ee%0Aq2QQkXT+1ecC3cyss5mVAAOA6ZkJ64u0dYVIAVm2DH72szBAvGlTOKNgzhz4+c8Lci1BnKSUEMxs%0AMjAb6G5mNWY2xN23AcOBGcBCYKq7L8heqCISa3//O1x1VVhL8NhjYRfSrVvD97QddV5IaR1CvtA6%0ABJGYeu+9sPncp5/uXEugIytzJtV1CLHoqFOXkUgMbd0KO/6A69IFLroI3nkHJk1SMshTsUgIGlQW%0AiZHt2+GJJ8IZBH36hCMszcICs8MP3+3TJTqxSAhqIYjEgDv88Y9wzDEwYEDYo/mJJ6Bt26gjkxTF%0AIiGohSASA4sXw6mnhsHjxx6DefPg9NM1YBwjsUgIIpKnFi2CBx4I97t3h2efhb/9LawlaNYs2tik%0A0WKRENRlJJJnli+HoUPDWoKRI0OrAMLJZc2bRxubNFksEoK6jETyxCefwNVXQ9euYbbQpZfCkiXw%0A1a9GHZlkQM72MhKRArBxI4wdC2efDTfdBJ07Rx2RZFAsEoKZlQPlXbt2jToUkeKydWs4uP6ll8J2%0A1PvvD++/H/YfkoKjLiMR+bId5xIcdlg4t3HZMtgxhqdkULBikRBEJIfeew9694ZzzgmbzU2fDrNm%0AwVe+EnVkkmWx6DISkRxYvz4cWN+uHey1Vxg01vTRoqKEIFLsFi+G666Dt96CBQugdWuYPTvqqCQC%0Asegy0joEkSz48MOw4VyPHmFB2cCBsG1b1FFJhGLRQnD3p4CnysrKLow6FpGC8M47cNxxIQFccklo%0AIey3X9RRScRi0UIQkQzYtAn+8pdw//DD4fLLwzYT995bkMmgthZOOAE++ijqSOJDCUGk0G3bBg89%0AFFYX9+0L69aFM4tvuSWcU1CgRo8Ok6NGjYo6kvjIWUIwsy5m9oiZTdtVmYhkiDtMmxbWEgwdCgce%0ACE8+CXvvHXVkWdWyZdhgddy4sJxi3Lhw3bJl1JHlv1TPVJ5oZivNbH5SeV8zW2RmVWY2Ylev4e7V%0A7j5kd2UikiFvvhm2mNhzz5AIXnst9KEUuOpqGDQIWrUK161aweDBsHRptHHFQaqDypOA+4Hf7Sgw%0As2bAWOBkoAaYa2bTgWbAmKTnX+DuK9OOVkR27Y03wjjBJZeEM4xnzIDvfa+o1hK0bw+lpbB5M7Ro%0AEW5LS8PyCtm1lFoI7j4T+HtScW+gKvFX/mfAFKC/u7/r7v2SvpQMRLJpyZKwsrisLHSeb9gQyk85%0ApaiSwQ4rVoQdN+bMCbcaWE5NOmMIBwDL6lzXJMrqZWZtzWw80NPMRjZUVs/zhppZpZlVrlq1Ko1w%0ARQrQypXhE+/QQ+Hpp8P00b/9LSwuK2IVFWFT1iOPDLcVFVFHFA85W4fg7muAYbsrq+d5E8ysFigv%0AKSk5OoshisTPhg3w+9+HpHDddeoXkbSk00JYDnSsc90hUZZx2u1UJGHTJrjzzjBKCuE8gpoauP9+%0AJQNJWzoJYS7Qzcw6m1kJMACYnpmwvkhbV0jR27YNHn4YunWDX/0KPv44HFYD2oVUMibVaaeTgdlA%0AdzOrMbMh7r4NGA7MABYCU919QfZCFSlS8+eHlcUXXggdO8LLL4e9h3bMqxTJEHP3qGNIWVlZmVdW%0AVkYdhkhufPJJ+Ot/7VooL4crroD+/cMqK5FGMLM33L1sd4+LxeZ2OkJTisq8eTByZBgbePttaNMG%0AZs6MOiopArHYy0iDylIUqqrCFtS9esHcufDTn4a9F0RyRC0EkXywY1uJkhK45hq46ioNFkvOqYUg%0AEpW1a3eeTNa7d0gEVVVhF1IlA4lALBKCSEHZvBnuuitsPf3DH8KWLWEDuptuChvxiEQkFglB6xCk%0AIGzbBhMnhrUEV14JxxwTpo82bx51ZCJATBKCuoykILz6KgwZAvvvD3/6Ezz/fBhAFskTsRhUFomt%0Al1+GhQvh4ovDoPFLL4VbrSWQPBSLFoK6jCR25s2D006DE0+E3/wGPvsslPfpo2QgeSsWCUFdRhIb%0A//d/4biuXr3CQTV33gnvvBOmk4rkOXUZiWTSxo3hXIKRI8MmdJo+KjESixaCSN769FO4/nr4yU/C%0A9T//MyxfDrfeGstkUFsbhjh0wlhxUkIQaYrNm+Huu8NagtGjwxjB1q3he3vvHW1saRg9GmbNglGj%0Aoo5EohCL3U7rbF1x4ZIlS6IOR4rd66/D2WeH8YJTToExY2I/fbRly5DjkrVoEc7kkXhLdbfTWLQQ%0ANKgskXOH1avD/S5dwuKyF1+EGTNinwwAqqvDWPiOIxZatQqHsi1dGm1cklsaVBbZnZkzYcSI0CX0%0A+uuw777wwgtRR5VR7dtDaWloJbRoEW5LS3UqZ7GJRQtBJBJvvw3f/34YZf3gAxg6tKC3o16xAoYN%0Agzlzwq0GlotPTlsIZtYFuBZo4+5nJcp+AJwOlAKPuPsfcxmTSL2efz4kgzZt4PbbYfjwgj+ysqJi%0A5/2xY6OLQ6KTcgvBzCaa2Uozm59U3tfMFplZlZmN2NVruHu1uw9JKnvS3S8EhgHnNCZ4kYxasSKc%0ASwBhhfHo0aFz/Ve/KvhkIAKN6zKaBPStW2BmzYCxwGlAD2CgmfUws2+Y2dNJX/vt5vWvS7yWSG7t%0AWEtw8MFhJPXzz8MOpNdeC/vsE3V0IjmTcpeRu880s05Jxb2BKnevBjCzKUB/dx8D9Evldc3MgNuA%0A59z9zVTjEUnbli0wblw4kGb16jCV9OaboVmzqCMTiUS6g8oHAMvqXNckyuplZm3NbDzQ08xGJop/%0ADpwEnGVmw+p5zlAzqzSzylWrVqUZrkgdL7wAl18ORx0VzjCeOhUOOSTqqEQik9NBZXdfQxgrqFt2%0AL3DvLp4zwcxqgfKSkpKjsxyiFDL3sM/Qhx/CRReFQePXXoNvfjPqyETyQrothOVAxzrXHRJlGaWF%0AaZK2WbPg+OPhjDPgwQfD9FEzJQOROtJNCHOBbmbW2cxKgAHA9PTD+iKdhyBNtngxlJeHZLB0aUgG%0Af/kL7KElOCLJGjPtdDIwG+huZjVmNsTdtwHDgRnAQmCquy/ITqgijbBjj64NG0K30G23wZIlYXHZ%0AXntFG5tInorF5nY7lJWVeWVlZdRhSD5buTLMGtq4ER56KJRt3Kh1BFLUCmpzO3UZyW6tWwc33hjW%0AEowdG6aO7thmQslAJCWxSAgaVJZdeumlkAhuuglOPRXmz4fx4zVOINJIsfiNUQtBvuTzz0P3EED3%0A7nDMMWEn0mnTwqllItJosUgIaiHIP+xYS3DUUXDWWeF6//3hmWdCUhCRJotFQhAB4NVX4TvfCdNI%0AN2+GSy+NOiKRghKLhKAuI2HqVPj2t6GqKuw/9Ne/wjnnhMVlIpIRsUgI6jIqUu+/v3M76n794K67%0AQkIYNkxrCUSyIBYJQYrMqlXwy1+GweKhQ8M4QatWcMUV0Lp11NGJFKxYJAR1GRWJdevC1NEuXeC+%0A++C888LJZeoWEsmJWCQEdRkVieeeC4vLTjkFFiwIK407dIg6KpGikdPtr0W+YPt2mDwZNm2Cn/0s%0ATCN9803o2TPqyESKUixaCFJg3OHZZ8MH/7nnhqTgHlYWKxmIREYJQXLrrbfghBPg9NPDTqSTJ8P/%0A/q/GCUTyQCwSggaVC8COXXU3bQrbUD/wACxcCAMGFMSeQ7W1Ic999FHUkYg0XSx+EzWoHGMffADn%0Anw/Dh4frb34zrC+4+OKCWkswenQ4lG3UqKgjEWm6WCQEiaHVq8O6gUMOgSlTYO+9d7YSmjePNrYM%0Aatky9HaNGxfGyMeNC9ctW0YdmUjjKSFI5k2fHtYS3HMPDB4cjrG87baCHCeoroZBg3YeudCqVajy%0A0qXRxiXSFDlLCGbWxcweMbNpdcoONbPxZjbNzC7OVSySBZ99trMD/YgjwrkE774LEyfCgQdGG1sW%0AtW8PpaVhr70WLcJtaSm0axd1ZCKNl1JCMLOJZrbSzOYnlfc1s0VmVmVmI3b1Gu5e7e5DksoWuvsw%0A4F+Bf2ls8JIHtm+Hxx8PZxCcf34o69QJ/vM/oUePKCPLmRUrwvZKc+aEWw0sS1ylujBtEnA/8Lsd%0ABWbWDBgLnAzUAHPNbDrQDBiT9PwL3H1lfS9sZmcAFwOPNSpyiZZ7WFk8ciS88w4ceSRcdlnUUUWi%0AomLn/bFjo4tDJF0pJQR3n2lmnZKKewNV7l4NYGZTgP7uPgbol2oA7j4dmG5mzwB/SPV5ErGHHw4b%0Az3XpAn/4Q9iKugCmj4oUs3S2rjgAWFbnugY4tqEHm1lb4Bagp5mNdPcxZtYHOBNoDjzbwPOGAkMB%0ADizgvuhYWLgQ1q6F444LCeDzz+GCC6CkJOrIRCQDcraXkbuvAYYllb0MvLyb500ws1qgvKSk5Ois%0ABSgNW7YMbrgBHn0Ujj02nFFQWho6zEWkYKTTxl8OdKxz3SFRlnFamBaRNWvgyiuhW7cwcPzLX4Yp%0ApSJSkNJJCHOBbmbW2cxKgAFAVj4ttHVFRP7nf+Duu8NE+yVLwoll++4bdVQikiUpdRmZ2WSgD7Cv%0AmdUAN7j7I2Y2HJhBmFk00d0XZC1Syb6tW8MZBK1ahSmk550Xtpo49NCoIxORHDDfsZ1ADJSVlXll%0AZWXUYRSe7dvhiSfg17+G996DH/0Ipk3b/fNEJBbM7A13L9vd42IxT1BdRln06qtw9NGhW6h1a3jm%0AmbCoTESKTiwSggaVs2D79nC7aVOYSvr738O8efD97xfknkMisnuxSAhqIWTQwoVw5plw1VXh+qST%0AYNGisCObFpaJFLVYfAKohZABy5bBkCFw+OHwwguw3347v1dA5xKISNPlbGGaROjxx0MycIdf/AKu%0AuQa+9rWooxKRPBOLFoK6jJpgw4ad226WlYWjKhcvDusKlAxEpB6xSAjqMmqErVvDsV1du8Ill4Sy%0A7t1h0iQ46KBIQxOR/BaLhCAp2L49HFV56KEhEXTtCv/2b1FHJSIxEouEoC6jFPzmNzBwYFhl/PTT%0AMHMm/IvOHBKR1Gmlcpy9/nq47d07bET33HMhKTRrFm1cIpJXCmqlsiRZtAjOOitsRX399aGsbVs4%0A91wlAxFpMiWEOFm+PJxSdthhMGMG3HijtpkQkYzROoQ4qagIs4WGD4drr9X0URHJqFgkBDMrB8q7%0Adu0adSi5tXEj3HsvdOwYtpYYOhTKy6FTp6gjE5ECFIsuo6Jbh7B1Kzz4YJg6OnIk/PnPobx584JK%0ABrW1cMIJO9fPiUi0YpEQisqMGWGMYNgw6NIFXnkFJkyIOqqsGD0aZs2CUaOijkREICZdRkVh2zbY%0Ac0/YsiW0BKZPh379CnIr6pYtYfPmndfjxoWvFi3CbtwiEo2ctRDMrIuZPWJm05LKW5tZpZn1y1Us%0AeWXu3LAF9U03hevycnjrrXBbgMkAoLo6nMfTqlW4btUqDJEsXRptXCLFLqWEYGYTzWylmc1PKu9r%0AZovMrMrMRuzqNdy92t2H1POtq4GpqYdcIBYtgrPPDovK3n4bOnQI5WYFv5agfXsoLQ2thBYtwm1p%0AKbRrF3VkIsUt1RbCJKBv3QIzawaMBU4DegADzayHmX3DzJ5O+trvyy8JZnYy8FdgZZNrEEcPPBDG%0ACZ5/Hm64IZxjfNFFUUeVUytWhGGSOXPCrQaWRaKX0hiCu880s05Jxb2BKnevBjCzKUB/dx8DpNr9%0A0wdoTUgom8zsWXffnuJz4+Xjj+Gzz+DrX4dvfStsQHfddV88qKaIVFTsvD92bHRxiMhO6YwhHAAs%0Aq3Ndkyirl5m1NbPxQE8zGwng7te6+y+BPwAP1ZcMzGxoYoyhctWqVWmEG5FNm+COO8KMoSuvDGVH%0AHRXWFxRpMhCR/JSzWUbuvgYY1sD3Ju3ieRPMrBYoLykpOTpL4WXetm3w29+G7SU+/BBOP33nOcYi%0AInkonRbCcqBjnesOibKMi+XCtFGjwsriTp3CVtRPPw1HHBF1VCIiDUqnhTAX6GZmnQmJYAAwKCNR%0AJYnN1hUvvgj77AO9esHFF4ejKwt4+qiIFJZUp51OBmYD3c2sxsyGuPs2YDgwA1gITHX3BdkLNY+9%0A8QacckpYT3D77aGsfXs44wwlAxGJDR2Qk44lS8JMoalTw3kE114bWgYtWkQdmYjIP6R6QE4stq7I%0A2y6jigp45hn49a/DDKLS0qgjEhFpMrUQGuOTT0KXUK9eYZXxxo2wbl1YWyAikqcK6ghNMys3swlr%0A166NJoC6awluuy2MGUDYhEfJQEQKRCwSQqTTTv/rv6BbN7j6ajjuOJg3LyQFEZECE4sxhJxzDwvL%0A9torHFbTsSM8/ng4zUVEpEDFooWQ0y6jl14KLYF///dwfc458NprSgYiUvBikRBy0mU0bx707Qvf%0A/W4423HHUZVmWksgIkUhFgkh68aMCTOH5s6Fu+6CxYth4MCooxIRyalYjCFkZR1CbW04iGa//eDE%0AE8OisquugjjtlyQikkGxaCFktMto7drw4d+1K1x/fSg77ji4+WYlAxEparFICBmxeXMYKO7SBW69%0ANewztON8ghiprQ3j2zphTEQyrXgSwtVXhy6hY46BN9+EyZNDKyFmRo+GWbPC7toiIplUPFtXfPAB%0AVFeH8YIYatkyNHKStWgRFlKLiDREW1ckO+ig2CYDCLls0KCwWwaE28GDYenSaOMSkcIRi4QQyxPT%0AMqx9+7CZ6ubNoVWweXO4btcu6shEpFDEIiFIsGIFDBsGc+aEWw0si0gmxWIdggQVFTvvjx0bXRwi%0AUphy1kIwsy5m9oiZTatT1sfMXjGz8WbWJ1exiIjIl6V6pvJEM1tpZvOTyvua2SIzqzKzEbt6DXev%0AdvchycXAeqAFUNOYwEVEJLNS7TKaBNwP/G5HgZk1A8YCJxM+zOea2XSgGTAm6fkXuPvKel73FXf/%0As5l9HfgNMLhx4YuISKaklBDcfaaZdUoq7g1UuXs1gJlNAfq7+xigX4qvuz1x92OgeSrPERGR7Ehn%0ADOEAYFmd65pEWb3MrK2ZjQd6mtnIRNmZZvYg8BihBVLf84aaWaWZVa5atSqNcEVEZFdyNsvI3dcA%0Aw5LKKoCK+p/xj8dMACZAWKmctQBFRIpcOi2E5UDHOtcdEmUZl9MT00REilQ6CWEu0M3MOptZCTAA%0AmJ6ZsEREJNdSnXY6GZgNdDezGjMb4u7bgOHADGAhMNXdF2QvVBERyabi2e1URKRIabdTERFplFgk%0ABO12KiKSfbFICGohiIhkXywSgloIIiLZF4uEICIi2ReLhKAuIxGR7ItFQlCXkYhI9sUiIYiISPbF%0AIiGoy0hEJPtikRDUZSQikn2xSAgiIpJ9SggiIgIoIYiISEIsEoIGlUVEsi8WCUGDyiIi2ReLhCAi%0AItmnhCAiIkAOE4KZdTGzR8xsWp2yPczsFjO7z8x+ks33r62FE06Ajz7K5ruIiMRXqmcqTzSzlWY2%0AP6m8r5ktMrMqMxuxq9dw92p3H5JU3B/oAGwFahoTeGONHg2zZsGoUdl8FxGR+Eq1hTAJ6Fu3wMya%0AAWOB04AewEAz62Fm3zCzp5O+9mvgdbsDr7n7FcDFTavCrrVsCWYwbhxs3x5uzUK5iIjslFJCcPeZ%0AwN+TinsDVYm//D8DpgD93f1dd++X9LWygZeuAT5O3P+8KRXYnepqGDQIWrUK161aweDBsHRpNt5N%0ARCS+0hlDOABYVue6JlFWLzNra2bjgZ5mNjJRXAGcamb3ATMbeN5QM6s0s8pVq1Y1Osj27aG0FDZv%0AhhYtwm1pKbRr1+iXEhEpaHvm6o3cfQ0wLKlsI5A8rpD8vAnABICysjJvynuvWAHDhsHQoTBhQhhg%0AFhGRL0onISwHOta57pAoyzgzKwfKu3bt2qTnV1TsvD92bGZiEhEpNOl0Gc0FuplZZzMrAQYA0zMT%0AloiI5Fqq004nA7OB7mZWY2ZD3H0bMByYASwEprr7gmwEqa0rRESyL6UuI3cf2ED5s8CzGY2oHul2%0AGYmIyO5p6woREQFikhDUZSQikn2xSAg6D0FEJPvMvUlT+yNhZmuBJXWK2gBrG7jecb9u2b7A6ia+%0AffJ7NeYx9ZWnEntD99Opx67iTOUx+VSXdOpR3/d2V7dC+flKvk6ui36+dh1jqo/Jp7p0c/fdd7G4%0Ae2y+gAmpXu+4n1RWman3bsxj6itPJfZd1KnJ9SikuqRTj1R+nlKMP3Y/X7uri36+svPzle91cfd4%0AdBnV8VQjrp9q4DGZeu/GPKa+8lRi39X9dBRKXdKpR33f213dCuXnK/k6znWJ089XfWX5VJd4dRml%0Ay8wq3b0s6jjSVSj1ANUlHxVKPUB1aay4tRDSNSHqADKkUOoBqks+KpR6gOrSKEXVQhARkYYVWwtB%0AREQaoIQgIiKAEoKIiCQUbULBZiLuAAAC5ElEQVQwsy5m9oiZTYs6lnSZ2Q/M7CEze8LMTok6nnSY%0A2aFmNt7MpplZVs7ZzhUza5047a9f1LGkw8z6mNkrif+XPlHHkw4z28PMbjGz+8zsJ1HHkw4zOz7x%0Af/Kwmb2WidcsqIRgZhPNbKWZzU8q72tmi8ysysxGAHg4C3qXp7VFqZF1edLdLyScSHdOFPHuSiPr%0AstDdhwH/CvxLFPE2pDH1SLgamJrbKFPTyLo4sB5oQTgqN680si79CYd5bSXmdXH3VxK/K08Dj2Yk%0AgHRWJObbF/AdoBcwv05ZM+A9oAtQArwN9Kjz/WlRx53ButwF9Io69nTrApwBPAcMijr2ptYDOJlw%0AaNT5QL+oY0+zLnskvv914PGoY0+zLiOAixKPybvf/Sb+3k8F9s7E+xdUC8HdZwJ/TyruDVR5aBF8%0ABkwh/JWQ1xpTFwtuB55z9zdzHevuNPb/xd2nu/tpwODcRrprjaxHH+A4YBBwoZnl1e9aY+ri7tsT%0A3/8YaJ7DMFPSyP+XGkI9AD7PXZSpaezvipkdCKx193WZeP90zlSOiwOAZXWua4BjzawtcAvQ08xG%0AuvuYSKJrnHrrAvwcOAloY2Zd3X18FME1UkP/L32AMwkfPFk/fCkD6q2Huw8HMLPzgdV1PlTzWUP/%0AJ2cCpwJfAe6PIrAmaOh35R7gPjM7HpgZRWBN0FBdAIYAv83UGxVDQqiXu68h9LnHnrvfC9wbdRyZ%0A4O4vAy9HHEbGuPukqGNIl7tXABVRx5EJ7r6R8CFaENz9hky+Xl41Y7NkOdCxznWHRFkcqS75p1Dq%0AAapLvspZXYohIcwFuplZZzMrIQz0TY84pqZSXfJPodQDVJd8lbu6RD2qnuER+slALTunlA1JlH8f%0AWEwYqb826jhVl3jWpVDqobrk71fUddHmdiIiAhRHl5GIiKRACUFERAAlBBERSVBCEBERQAlBREQS%0AlBBERARQQhARkQQlBBERAZQQREQk4f8BxYXjZrwZKWwAAAAASUVORK5CYII=%0A)

## Computing dot products<a href="#Computing-dot-products" class="anchor-link">¶</a>

Let \$x\$ and \$y\$ be two vectors of length \$n\$, and denote their dot
product by \$f(x, y) \equiv x^T y\$.

Now suppose we store the values of \$x\$ and \$y\$ *exactly* in two
Python arrays, `x[0:n]` and `y[0:n]`. Further suppose we compute their
dot product by the program, `alg_dot()`.

In \[10\]:

    def alg_dot (x, y):
        p = [xi*yi for (xi, yi) in zip (x, y)]
        s = alg_sum (p)
        return s

**Exercise 1** (OPTIONAL -- 0 points, not graded or collected). Show
under what conditions `alg_dot()` is backward stable.

> *Hint.* Let \$(x_k, y_k)\$ denote the exact values of the
> corresponding inputs, \$(\mathtt{x}\[k\], \mathtt{y}\[k\])\$. Then the
> true dot product, \$x^T y = \sum\_{l=0}^{n-1} x_l y_l\$. Next, let
> \$\hat{p}\_k\$ denote the \$k\$-th computed product, i.e.,
> \$\hat{p}\_k \equiv x_k y_k (1 + \gamma_k)\$, where \$\gamma_k\$ is
> the \$k\$-th round-off error and \$\|\gamma_k\| \leq \epsilon\$. Then
> apply the results for `alg_sum()` to analyze `alg_dot()`.

**Answer.** Following the hint, `alg_sum` will compute
\$\hat{s}\_{n-1}\$ on the *computed* inputs, \$\\{\hat{p}\_k\\}\$. Thus,

\$\$ \begin{eqnarray} \hat{s}\_{n-1} & \approx & \sum\_{l=0}^{n-1}
\hat{p}\_l (1 + \Delta_l) \\\\ & = & \sum\_{l=0}^{n-1} x_l y_l (1 +
\gamma_l) (1 + \Delta_l) \\\\ & = & \sum\_{l=0}^{n-1} x_l y_l (1 +
\gamma_l + \Delta_l + \gamma_l \Delta_l). \end{eqnarray} \$\$

Mathematically, this appears to be the exact dot product to an input in
which \$x\$ is exact and \$y\$ is perturbed (or vice-versa). To argue
that `alg_dot` is backward stable, we need to establish under what
conditions the perturbation, \$\left\| \gamma_l + \Delta_l + \gamma_l
\Delta_l \right\|\$, is "small." Since \$\|\gamma_l\| \leq \epsilon\$
and \$\|\Delta_l\| \leq n \epsilon\$,

\$\$ \left\| \gamma_l + \Delta_l + \gamma_l \Delta_l \right\| \leq \|
\gamma_l \| + \| \Delta_l \| + \|\gamma_l\| \cdot \|\Delta_l\| \leq
(n+1) \epsilon + \mathcal{O}(n \epsilon^2) \approx (n+1) \epsilon. \$\$

## More accurate summation<a href="#More-accurate-summation" class="anchor-link">¶</a>

Suppose you wish to compute the sum, \$s = x_0 + x_1 + x_2 + x_3\$.
Let's say you use the "standard algorithm," which accumulates the terms
one-by-one from left-to-right, as done by `alg_sum()` above.

For the standard algorithm, let the \$i\$-th addition incur a roundoff
error, \$\delta_i\$. Then our usual error analysis would reveal that the
absolute error in the computed sum, \$\hat{s}\$, is approximately:

\$\$ \begin{array}{rcl} \hat{s} - s & \approx & x_0(\delta_0 +
\delta_1 + \delta_2 + \delta_3) + x_1(\delta_1 + \delta_2 + \delta_3) +
x_2(\delta_2 + \delta_3) + x_3\delta_3. \end{array} \$\$

And since \$\|\delta_i\| \leq \epsilon\$, you would bound the absolute
value of the error by,

\$\$ \begin{array}{rcl} \left\| \hat{s} - s \right\| & \lesssim &
(4\|x_0\| + 3\|x_1\| + 2\|x_2\| + 1\|x_3\|)\epsilon. \end{array} \$\$

Notice that \$\|x_0\|\$ is multiplied by 4, \$\|x_1\|\$ by 3, and so on.

In general, if there are \$n\$ values to sum, the \$\|x_i\|\$ term will
be multiplied by \$n-i\$.

**Exercise 2** (3 points). Based on the preceding observation, implement
a new summation function, `alg_sum_accurate(x)` that computes a more
accurate sum than `alg_sum()`.

> *Hint 1.* You do **not** need `Decimal()` in this problem. Some of you
> will try to use it, but it's not necessary.
>
> *Hint 2.* Some of you will try to "implement" the error formula to
> somehow compensate for the round-off error. But that shouldn't make
> sense to do. (Why not? Because the formula above is a *bound*, not an
> exact formula.) Instead, the intent of this problem is to see if you
> can look at the formula and understand how to interpret it. That is,
> what does the formula tell you?

In \[11\]:

    def alg_sum_accurate(x):
        assert type(x) is list
        ### BEGIN SOLUTION
        # Idea: Each term of the error bound is $(n-i) |x_i| \epsilon$.
        # In other words, lower values of $i$ have a larger coefficient $n-i$.
        # Therefore, one idea is simply to _sort_ the data in increasing order
        # of **magnitude** (absolute value), so that the larger coefficients
        # are paired with the smaller values.
        x_sorted = sorted(x, key=abs)
        return sum(x_sorted)
        ### END SOLUTION

In \[12\]:

    # Test: `alg_sum_accurate_test`
    from math import exp
    from numpy.random import lognormal

    print("Generating non-uniform random values...")
    N = [10, 10000, 10000000]
    x = [lognormal(-10.0, 10.0) for _ in range(max(N))]
    print("Range of input values: [{}, {}]".format(min(x), max(x)))

    print("Computing the 'exact' sum. May be slow so please wait...")
    x_exact = [Decimal(x_i) for x_i in x]
    s_exact = [float(sum(x_exact[:n])) for n in N]
    print("==>", s_exact)

    print("Running alg_sum()...")
    s_alg = [alg_sum(x[:n]) for n in N]
    print("==>", s_alg)

    print("Running alg_sum_accurate()...")
    s_acc = [alg_sum_accurate(x[:n]) for n in N]
    print("==>", s_acc)

    print("Summary of relative errors:")
    ds_alg = [abs(s_a - s_e) / s_e for s_a, s_e in zip(s_alg, s_exact)]
    ds_acc = [abs(s_a - s_e) / s_e for s_a, s_e in zip(s_acc, s_exact)]
    display (pd.DataFrame ({'n': N,
                            'rel_err(alg_sum)': ds_alg,
                            'rel_err(alg_sum_accurate)': ds_acc}))

    assert all([r_acc < r_alg for r_acc, r_alg in zip(ds_acc[1:], ds_alg[1:])]), \
           "The 'accurate' algorithm appears to be less accurate than the conventional one!"

    print("\n(Passed!)")

    Generating non-uniform random values...
    Range of input values: [2.4988313527996013e-28, 4.508417815653759e+17]
    Computing the 'exact' sum. May be slow so please wait...
    ==> [0.34441634937208654, 59286881538197.42, 1.5616792932459144e+18]
    Running alg_sum()...
    ==> [0.34441634937208654, 59286881538197.09, 1.5616792932375094e+18]
    Running alg_sum_accurate()...
    ==> [0.34441634937208654, 59286881538197.42, 1.5616792932459144e+18]
    Summary of relative errors:

|     | n        | rel_err(alg_sum) | rel_err(alg_sum_accurate) |
|-----|----------|------------------|---------------------------|
| 0   | 10       | 0.000000e+00     | 0.0                       |
| 1   | 10000    | 5.534530e-15     | 0.0                       |
| 2   | 10000000 | 5.382022e-12     | 0.0                       |

    (Passed!)

**Done!** You have reached the end of Part 1. There are no additional
parts, so if you are satisfied, be sure to submit both parts, declare
victory, and move on!